In [1]:
import json
import os
os.chdir("/home/ubuntu/work/somin")
os.getcwd()
from pathlib import Path

data_path = Path('evaluation/dataset/')
with open(data_path / 'goldset_review1.json', 'r') as f:
    review1 = json.load(f)
with open(data_path / 'goldset_review2.json', 'r') as f:
    review2 = json.load(f)

In [ ]:
with open(data_path / 'goldset_final.json', 'r') as f:
    goldset_final = json.load(f)

In [ ]:
# review1, review2 에서 corrected_grade 0, 1 인 isbn의 것만 goldset_final에 final_grade 0, 1로 업데이트한 항목을 새로 만들어줘.
for review in review1 + review2:
    if review['corrected_grade'] in [0, 1]:
        for item in goldset_final:
            if item['isbn'] == review['isbn']:
                item['final_grade'] = review['corrected_grade']
                break


In [4]:
with open(data_path / 'goldset_review1.json') as f:
    r1 = json.load(f)
with open(data_path / 'goldset_review2.json') as f:
    r2 = json.load(f)
with open(data_path / 'goldset_final.json') as f:
    gf = json.load(f)

# 0 또는 1인 항목만 추출
r1_filtered = [x for x in r1 if x.get('corrected_grade') in [0, 1]]
r2_filtered = [x for x in r2 if x.get('corrected_grade') in [0, 1]]

# goldset_final의 (query_id, isbn) 키 확인
gf_keys = {(x['query_id'], x['isbn']): i for i, x in enumerate(gf)}

# review1에서 매칭되는지 확인
matched_r1 = []
unmatched_r1 = []
for item in r1_filtered:
    key = (item['query_id'], item['isbn'])
    if key in gf_keys:
        matched_r1.append(item)
    else:
        unmatched_r1.append(item)

matched_r2 = []
unmatched_r2 = []
for item in r2_filtered:
    key = (item['query_id'], item['isbn'])
    if key in gf_keys:
        matched_r2.append(item)
    else:
        unmatched_r2.append(item)

print('review1 matched:', len(matched_r1), 'unmatched:', len(unmatched_r1))
print('review2 matched:', len(matched_r2), 'unmatched:', len(unmatched_r2))

if unmatched_r1:
    print('unmatched r1:', [(x['query_id'], x['isbn']) for x in unmatched_r1])
if unmatched_r2:
    print('unmatched r2:', [(x['query_id'], x['isbn']) for x in unmatched_r2])

# review1과 review2에서 겹치는 (query_id, isbn) 확인
r1_keys = {(x['query_id'], x['isbn']) for x in r1_filtered}
r2_keys = {(x['query_id'], x['isbn']) for x in r2_filtered}
overlap = r1_keys & r2_keys
print('review1 & review2 overlap:', overlap)

review1 matched: 38 unmatched: 0
review2 matched: 20 unmatched: 0
review1 & review2 overlap: set()


In [5]:
from copy import deepcopy

corrections = {}
for item in r1 + r2:
    if item.get('corrected_grade') in [0, 1]:
        key = (item['query_id'], item['isbn'])
        corrections[key] = item['corrected_grade']

print(f'총 수정 대상: {len(corrections)}개')
print('corrected_grade=0:', sum(1 for v in corrections.values() if v == 0))
print('corrected_grade=1:', sum(1 for v in corrections.values() if v == 1))

# goldset_final 전체를 복사하되 해당 항목의 final_grade만 업데이트
updated = deepcopy(gf)
count = 0
for item in updated:
    key = (item['query_id'], item['isbn'])
    if key in corrections:
        old = item['final_grade']
        item['final_grade'] = corrections[key]
        print(f'[{item["query_id"]}] {item["isbn"]} {item["title"][:20]} | final_grade: {old} → {corrections[key]}')
        count += 1

print(f'\n업데이트 완료: {count}개')

with open(data_path / 'goldset_final_corrected.json', 'w', encoding='utf-8') as f:
    json.dump(updated, f, ensure_ascii=False, indent=2)

print('저장 완료: goldset_final_corrected.json')

총 수정 대상: 58개
corrected_grade=0: 19
corrected_grade=1: 39
[S16] 9788941336624 항상, 봄 | final_grade: 2 → 1
[S18] 9788946423084 시가 되는 순간들 - 이제야 산문집 | final_grade: 2 → 1
[S06] 9788956582955 내 인생에 농담 걸기 | final_grade: 2 → 1
[S07] 9788957821886 존리와 함께 떠나는 부자 여행 1 : | final_grade: 2 → 1
[S09] 9788959252398 달라재로 가는 길 - 정달자 자전소설 | final_grade: 2 → 0
[S19] 9788959256891 꿈꾸는 강 - 이야기식 에세이 | final_grade: 2 → 0
[S20] 9788959793013 진짜 수학을 못하는 애들을 위한 수학 | final_grade: 2 → 0
[S10] 9788963391816 어느 필부, 삶의 소나티네 | final_grade: 2 → 0
[S10] 9788963490816 소리 없는 빛의 노래 | final_grade: 2 → 1
[S06] 9788965180272 사람의 마음이 읽힌다 - 나를 숨기고 | final_grade: 2 → 1
[S02] 9788966370214 시장경제의 약속 - 그래도 성장이 희 | final_grade: 2 → 1
[S18] 9788967993641 내가 사랑하는 것들은 왜 빨리 사라질 | final_grade: 2 → 1
[S17] 9788972974758 나의 라임 오렌지나무 - MBC 느낌 | final_grade: 2 → 0
[S06] 9788977182400 그리운 사람에게 주고 싶은 책 | final_grade: 2 → 1
[S08] 9788977282827 십팔사략 1 - 춘추 전국시대 = 十 | final_grade: 2 → 1
[S09] 9788979211238 야누스의 도시 | final_grade: 2 → 0
[S18] 978898

In [6]:
import pandas as pd

# goldset_final_corrected.json 로드
with open(data_path / 'goldset_final_corrected.json') as f:
    goldset_corrected = json.load(f)

# DataFrame 생성 + gold_status 부여 (corrected 파일은 전부 included)
df = pd.DataFrame(goldset_corrected)
df['gold_status'] = df['final_grade'].apply(lambda x: 'included' if x is not None else 'excluded')

print(f'총 항목 수: {len(df):,}')
print(f'  included: {(df["gold_status"]=="included").sum():,}')
print(f'  excluded: {(df["gold_status"]=="excluded").sum():,}')

print("\n=== confidence 분포 ===")
print(df["confidence"].value_counts().to_string())

print("\n=== final_grade 분포 (포함된 문서) ===")
included = df[df["gold_status"] == "included"]
print(included["final_grade"].value_counts().sort_index().to_string())

print("\n=== query_id별 포함/제외 ===")
summary = df.groupby("query_id")["gold_status"].value_counts().unstack(fill_value=0)
summary["total"] = summary.sum(axis=1)
print(summary.to_string())

# ── query_id별 grade 분포 상세 테이블 ──────────────────────────────────
print("\n[query_id별 relevance_grade 분포]")

df["_label"] = df.apply(
    lambda r: f"grade {int(r['final_grade'])}" if r["gold_status"] == "included" else "excluded",
    axis=1,
)
pivot = df.groupby(["query_id", "_label"]).size().unstack(fill_value=0)
df.drop(columns=["_label"], inplace=True)

grade_cols = [f"grade {i}" for i in range(4) if f"grade {i}" in pivot.columns]
col_order  = grade_cols + (["excluded"] if "excluded" in pivot.columns else [])
pivot      = pivot.reindex(columns=col_order, fill_value=0)
pivot["total"] = pivot.sum(axis=1)

col_w = 9
qid_w = 10
header = f"  {'query_id':>{qid_w}}" + "".join(f"  {c:>{col_w}}" for c in col_order) + f"  {'total':>{col_w}}"
sep    = "-" * (len(header))
print(header)
print(sep)
for qid, row in pivot.iterrows():
    vals = "".join(f"  {int(row[c]):>{col_w}}" for c in col_order)
    print(f"  {qid:>{qid_w}}{vals}  {int(row['total']):>{col_w}}")


총 항목 수: 2,519
  included: 2,519
  excluded: 0

=== confidence 분포 ===
confidence
HIGH      1602
MEDIUM     917

=== final_grade 분포 (포함된 문서) ===
final_grade
0     260
1    1679
2     380
3     200

=== query_id별 포함/제외 ===
gold_status  included  total
query_id                    
S01               113    113
S02               115    115
S03                85     85
S04               102    102
S05               130    130
S06               119    119
S07                67     67
S08               125    125
S09               136    136
S10               114    114
S11               141    141
S12               122    122
S13               136    136
S14               129    129
S15               114    114
S16               144    144
S17               114    114
S18               121    121
S19               131    131
S20               131    131
S21               130    130

[query_id별 relevance_grade 분포]
    query_id    grade 0    grade 1    grade 2    grade 3      total
-------------